# 2 · Experten: Rolling-Origin-Prognosen (Stufe 1)

Jeder Experte prognostiziert für jeden Ursprung $t$ (letzter bekannter Tag) und Horizont $h$ die RV am Tag $t+h$, als Mittelwert und 21 Quantile. Er sieht nur Daten bis einschliesslich $t$.

Die Prognosen sind teuer (Foundation Models) und werden pro Serie und Modell als `<SID>__<Modell>.parquet` gecacht. **Vorhandene Dateien werden nur geladen, nie überschrieben.** Für `binance` liegen Chronos-2 und TimesFM-2.5 schon in `results/`.

| Experte | Input | Idee |
|---|---|---|
| **RW** | log-RV | „morgen wie heute“, Streuung aus historischen h-Tages-Änderungen |
| **EWMA** | log-RV | exponentiell geglättete log-RV (Halbwertszeit 5 Tage) |
| **HAR** | log-RV | $\ln RV_{t+h} = \beta_0 + \beta_1 \ln RV_t + \beta_2 \ln RV^{(5)}_t + \beta_3 \ln RV^{(22)}_t$, OLS im 1000-Tage-Fenster |
| **GARCH** | Renditen | GARCH(1,1)-t, über Log-Residuen auf die RV kalibriert |
| **Chronos-2-full** | log-RV | Foundation Model (Amazon), zero-shot, Kontext 1024 |
| **TimesFM-2.5** | log-RV | Foundation Model (Google), zero-shot, 9 Quantile → 21 ergänzt |

Ökonometrische Modelle: Quantile aus den empirischen Log-Residuen, Mittelwert per Smearing (Duan 1983). Foundation Models: Quantile direkt, Mittelwert durch Integration der Quantile.

| **t0-gamma** | log-RV | Foundation Model (The Forecasting Company, Sept. 2026), zero-shot, Kontext 1024 |
| **Timestral-209M** | log-RV | Foundation Model (Mistral AI, Okt. 2026), zero-shot, Kontext 1024 |

**t0 und Timestral** sind nach unserer Testperiode erschienen und könnten die Daten im Pretraining gesehen haben (Kontamination). Ihre Historie wird hier gerechnet, damit sie für Analysen und später für das Gate bereitsteht. Im Standard-Pool (`03_features`) sind sie **nicht**. Fehlt das Paket, wird der Experte übersprungen (Installation siehe README).

**Der Code der Experten steht in `src/pipeline/experts.py`.** Die App verwendet denselben Code, ihre Prognosen sind deshalb exakt dieselben wie hier. Gemeinsame Einstellungen (Horizonte, Schätzfenster, Kontext) stehen in `src/pipeline/common.py`.

**Neuen Experten hinzufügen:** Klasse mit `name` und `forecast(s, origins)` in `pipeline/experts.py` schreiben, in `EXPERTS` eintragen, Namen hier in `MODELS`.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATASET = "binance"        # Name aus 01_daten.ipynb, z.B. "binance" oder "syn_regime_signal_s0"

DATA_DIR, RESULTS_DIR = Path("data"), Path("results")
INFO = json.loads((DATA_DIR / f"{DATASET}.json").read_text())
DAILY_DIR, FORECAST_DIR = Path(INFO["daily_dir"]), Path(INFO["forecast_dir"])
SIDS = INFO["sids"]
SYNTHETIC = INFO["info"]["source"] == "synthetic"
print(f"Datensatz {DATASET}: {len(SIDS)} Serien {SIDS}")

# Gemeinsamer Code mit der App: src/pipeline/ (Einstellungen für beide in pipeline/common.py)
from pipeline.common import QUANTILES, QCOLS, LV as _LV, HORIZONS
from pipeline import common as C, experts as ex

import time

MODELS = ["RW", "EWMA", "HAR", "GARCH", "Chronos-2-full", "TimesFM-2.5",
          "t0-gamma", "Timestral-209M"]      # t0 und Timestral: nur Historie, nicht im Pool (Kontamination)
SKIP_MISSING_PACKAGES = True    # Foundation Model nicht installiert -> überspringen statt abbrechen

# gemeinsam mit der App, Änderung in pipeline/common.py (gespeicherte Prognosen danach löschen)
print(f"Horizonte {C.HORIZONS}, Historie {C.MIN_HISTORY}, Schätzfenster {C.EST_WINDOW}, "
      f"GARCH-Refit alle {C.GARCH_REFIT_EVERY}, Kontext {C.CONTEXT}, Batch {C.FM_BATCH}")

Datensatz binance: 3 Serien ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']


/Users/sebastianjung/Documents/HSG/HS 26/BP/Learning_to_Forecast_Time-Series_with_Multiple_Experts/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Serien und Ursprünge

In [2]:
series = {sid: ex.RVSeries(sid, pd.read_parquet(DAILY_DIR / f"{sid}_daily.parquet")) for sid in SIDS}
for s in series.values():
    print(f"{s.sid}: {len(s.origins)} Ursprünge ab {s.dates[s.origins[0]].date()}")

BTCUSDT: 1922 Ursprünge ab 2021-05-21
ETHUSDT: 1922 Ursprünge ab 2021-05-21
SOLUSDT: 1702 Ursprünge ab 2021-12-29


## Ökonometrische Experten

`RandomWalk`, `EWMA`, `HAR` und `GARCH` in `pipeline/experts.py`. Quantile aus den empirischen Log-Residuen im 1000-Tage-Fenster, Mittelwert per Smearing. GARCH wird alle 5 Ursprünge neu geschätzt, dazwischen mit den alten Parametern auf die neuen Daten angewendet.

In [3]:
from pipeline.experts import RandomWalk, EWMA, HAR, GARCH      # Code: src/pipeline/experts.py
from pipeline.common import dist_from_log_resid, mean_from_quantiles

## Foundation Models

Alle laufen zero-shot auf der log-RV mit 1024 Tagen Kontext. Sie werden erst importiert, wenn sie wirklich rechnen müssen.

- **Chronos-2:** `cross_learning=False`, sonst teilen Kontexte im Batch Information (Leakage).
- **TimesFM-2.5** liefert nur die Quantile 0.1 … 0.9. Die übrigen werden **in Logs** auf der Normal-Skala ergänzt: innen linear, an den Rändern mit der lokalen Steigung. Das ist exakt, falls die log-RV normalverteilt ist (Kontrolle unten).
- **t0-gamma** und **Timestral-209M** liefern direkt unsere 21 Quantile. Kürzere Kontexte (frühe Ursprünge) werden links mit NaN aufgefüllt, beide behandeln das als fehlend. `symmetric_averaging=True`, `clip_nonnegative=False` (log-RV ist negativ).

In [4]:
from statistics import NormalDist

from pipeline.experts import Chronos2, TimesFM25, T0Gamma, Timestral, expand_quantiles, NATIVE9

ex.DEVICE = None    # None = cuda falls verfügbar, sonst cpu

nd = NormalDist(-7.0, 0.6)    # Kontrolle: bei normalverteilter log-RV muss die Ergänzung exakt sein
print("max. Fehler der Quantil-Ergänzung bei Normalverteilung:",
      float(np.abs(expand_quantiles(np.array([nd.inv_cdf(p) for p in NATIVE9]))
                   - np.array([nd.inv_cdf(p) for p in QUANTILES])).max()))

EXPERTS = ex.EXPERTS
print("verfügbare Experten:", list(EXPERTS))

max. Fehler der Quantil-Ergänzung bei Normalverteilung: 8.881784197001252e-16


## Prognosen erzeugen (mit Cache)

Eine Zeile pro Ursprung und Horizont mit Ziel `y` (gemessene RV), `mean` und den 21 Quantilen. Laufzeiten landen in `runtimes.csv` im Prognose-Ordner.

In [5]:
def forecast_path(sid, model):
    return FORECAST_DIR / f"{sid}__{model}.parquet"


def run_expert(expert, s):
    t0 = time.time()
    mean, q = expert.forecast(s, s.origins)
    runtime = time.time() - t0
    ex.forecast_frame(expert.name, s, s.origins, mean, q).to_parquet(forecast_path(s.sid, expert.name))
    with open(FORECAST_DIR / "runtimes.csv", "a") as f:
        f.write(f"{s.sid},{expert.name},{runtime:.1f}\n")
    print(f"{expert.name:>16} | {s.sid}: {runtime:.0f}s")


FORECAST_DIR.mkdir(parents=True, exist_ok=True)
for m in MODELS:
    todo = [sid for sid in SIDS if not forecast_path(sid, m).exists()]
    if not todo:
        print(f"{m:>16} | gecacht")
        continue
    try:
        expert = EXPERTS[m]()
    except ImportError as e:
        if not SKIP_MISSING_PACKAGES:
            raise
        print(f"{m:>16} | übersprungen, Paket fehlt ({e.name})")
        continue
    for sid in todo:
        run_expert(expert, series[sid])
    del expert

              RW | gecacht
            EWMA | gecacht
             HAR | gecacht
           GARCH | gecacht
  Chronos-2-full | gecacht
     TimesFM-2.5 | gecacht


## Überblick: welche Prognosen gibt es?

In [6]:
rows = []
for p in sorted(FORECAST_DIR.glob("*__*.parquet")):
    sid, m = p.stem.split("__")
    if sid in SIDS:
        f = pd.read_parquet(p, columns=["h", "origin"])
        rows.append({"Serie": sid, "Modell": m, "Prognosen (h=1)": int((f["h"] == 1).sum()),
                     "von": f["origin"].min().date(), "bis": f["origin"].max().date()})
display(pd.DataFrame(rows).pivot_table(index="Modell", columns="Serie", values="Prognosen (h=1)"))

Serie,BTCUSDT,ETHUSDT,SOLUSDT
Modell,,,
Chronos-2,1922.0,1922.0,1702.0
Chronos-2-full,1922.0,1922.0,1702.0
EWMA,1922.0,1922.0,1702.0
GARCH,1922.0,1922.0,1702.0
HAR,1922.0,1922.0,1702.0
RW,1922.0,1922.0,1702.0
TimesFM-2.5,1922.0,1922.0,1702.0
